<h1 style="color: #336699;"><b>MGC Spatial course</b></h1>

<h2 style="color: #336699;">Spatially-resolved transcriptomics data integration</h2>
<h3 style="color: #336699;">Day 3 - Practical 5</h3>

<h4>12th November 2025</h4>
<h4>Benedetta Manzato (LUMC)</h4>

<div style="border: 2px solid #336699; padding: 15px; border-radius: 8px;">
  <span style="color: #336699; font-size: 24px; font-weight: bold;">Agenda:</span>
  <ol style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #333333;">
    <li>Dataset description</li>
    <li>Spatial imputation with SpaGE and Tangram</li>
    <li>Cell deconvolution with Cell2location</li>
  </ol>
</div>



In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Importing necessary libraries for spatial data analysis and visualization
import os, re
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import random
import numpy as np
import anndata as ad
from scipy.sparse import csc_matrix

# Display the version of important packages for reproducibility
sc.logging.print_header()

# Set the default aesthetics for Scanpy plots
sc.set_figure_params(facecolor='white', figsize=(8, 8))  # White background, figure size 8x8
sc.settings.verbosity = 0  # Set verbosity to show only errors (0) # errors (0), warnings (1), info (2), hints (3)

<font size="3"> We create an output folder in your local space to save outputs:

In [ ]:
folder_path = './outputs/'
if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Folder created: {folder_path}")
else:
    print(f"Folder already exists: {folder_path}")

### Dataset: MERFISH Spatial Transcriptomics of Adult Mouse Brain (Zhuang‑ABCA‑1)

<font size="3">We will use the **Zhuang-ABCA-1 dataset**: 4.2 million cell spatial transcriptomics dataset spanning 147 coronal sections with a [....] gene panel. 2.8 million cells passed cell classification confidence score threshold and displayed in the ABC atlas.

<font size="3">We decided to work with **one section only**, the one that captures most cells (section 80).

<img src="https://media.springernature.com/lw685/springer-static/image/art%3A10.1038%2Fs41586-023-06808-9/MediaObjects/41586_2023_6808_Fig1_HTML.png?as=webp" width="800" height="300">

### Read data 

<font size="3">We will read the AnnData object that has been filtered to have only section 80 and QC was performed from the authors of the study.

In [ ]:
adata_section = ad.read_h5ad(f"data/adata_section80.h5ad")

<font size="3"> The dataset was obtain with a <strong>targeted / imaging-based</strong> technology and not the whole transcriptome is measured, just a limited gene panel.

<div style="border: 2px solid #c1121f; padding: 15px; border-radius: 8px;">
<ul style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #c1121f; list-style-type: none; font-weight: bold;">
<li>Question: how many genes are measured?</li>
</ul>
</div>

<font size="3"> <strong>Reminder from Practical 0:</strong> Spatial coordinates, being metadata about cells/spots are stored either in **`.obs`** or **`.obsm`** Spot metadata also contains info about class, subclass, neurotransmitter, patient metadata and section.

In [ ]:
adata_section.obs.head()

<font size="3"> Some spatial tools require the spatial coordinates can be stored as `spatial` or `X_spatial`

In [ ]:
# Add X_spatial to the right field
x_spatial = np.array(adata_section.obs[['x','y']])

adata_section.obsm['X_spatial'] = x_spatial
adata_section.obsm['X_spatial']

In [ ]:
import gc
# collect garbage
gc.collect()

# Prediction of spatial patterns for unmeasured genes using scRNA-seq data with SpaGE
[SpaGE: Spatial Gene Enhancement using scRNA-seq (Abdelaal et al. 2020)](https://academic.oup.com/nar/article/48/18/e107/5909530)

In [ ]:
from SpaGE import SpaGE
import scipy.stats as st

<figure>
    <img src="https://oup.silverchair-cdn.com/oup/backfile/Content_public/Journal/nar/48/18/10.1093_nar_gkaa740/2/m_gkaa740fig1.jpeg?Expires=1763574433&Signature=q5iD6GJ52bxxa1tzaiwlz0kkfB~GmNg8-Dz2CpsvJCDxh7FqSq40dscOLsR1~N8-39VFIr9z-X9W2Iagd8jrZeWU-Fi3LuQy-~~nfvGbcAgJTSHUFcWyiEQwHrtU6Q4Msbr-EJ4NxOhzLI~hW93T3VSnWFeREXrelpxSnsg1g10N5~yDN6ELmC0eogFO9pDy7OtlfdqVRstgCM6maeBsXqPztIWttluTRrhC6e2IrfDvHYZc2aD8pPor~frp2lxciymsT0jaRBdR5Fi0JsqUso9kpuyBtcmaRsIbOoxAybZIjq9IMNbZhAfLTJijOvu~ITfdmJNZecU7edpNyjrvyQ__&Key-Pair-Id=APKAIE5G5CRDK6RD3PGA" width="700" height="280" alt="Description of the image">
    <figcaption>From <a href="https://academic.oup.com/nar/article/48/18/e107/5909530" target="_blank">Abdelaal et al. (2020) NAR</a></figcaption>
</figure>

<font size="3"> SpaGE takes as input the (targeted) spatial dataset (query) and a single-cell RNA dataset used as reference. The two datasets should be generated under the same conditions (same species, tissue, region) as much as possible.

<font size="3"> We will use [Mouse whole-brain transcriptomic cell type atlas (Hongkui Zeng)](https://www.nature.com/articles/s41586-023-06812-z).This scRNAseq dataset profiled 7 million cells (approximately 4.0 million cells passing quality control). More in detail:

 - <font size="3"> 1.7 million single cell transcriptomes spanning the whole adult mouse brain using 10Xv2 chemistry (WMB-10Xv2)

 - <font size="3"> 2.3 million single cell transcriptomes spanning the whole adult mouse brain using 10Xv3 chemistry (WMB-10Xv3)

 - <font size="3"> 1687 single cell transcriptomes spanning the whole adult mouse brain using the 10X Multiome chemistry (WMB-10XMulti)

<figure>
    <img src="https://media.springernature.com/lw685/springer-static/image/art%3A10.1038%2Fs41586-023-06812-z/MediaObjects/41586_2023_6812_Fig1_HTML.png?as=webp" width="700" height="280" alt="Description of the image">
    <figcaption>From <a href="https://www.nature.com/articles/s41586-023-06812-z" target="_blank">Yao et al. (2023) Nature Methods</a></figcaption>
</figure>


<font size="3"> The size is of the combined single-cell object from AbcProjectCache is around 280GB. For this reason, the three objects have been previously downloaded, concatenated, preprocessed and subsetted to obtain 28981 cells. For your knowledge, you can find all the steps in `obtain_scref.py`.

<font size="3"> Let's read the resulting single-cell AnnData object:

In [ ]:
adata_sc = ad.read_h5ad(f"data/sc_adata_1percent.h5ad")

<font size="3"> Inspect the object:

In [ ]:
adata_sc

In [ ]:
adata_sc.obs.head()

<font size="3"> In this case the feature metadata are stored in a separate csv. We need to read read and then add them to the right field in the AnnData object

<div style="border: 2px solid #c1121f; padding: 15px; border-radius: 8px;">
<ul style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #c1121f; list-style-type: none; font-weight: bold;">
<li>Add feature metadata to the AnnData object</li>
</ul>
</div>

<font size="3"> We previously made sure that the genes in the csv file are sorted in the same way as the ones already present in the AnnData object, so you will just need to add it to the right field now)

In [ ]:
# read 
genes = pd.read_csv(f"data/gene.csv",index_col=0)

genes.head()

In [ ]:
# add
# ....

In [ ]:
# Reset the index to turn the current index into a column
adata_sc.var.reset_index(inplace=True)
adata_sc.var.set_index('gene_symbol', inplace=True)
adata_sc.var.head()

In [ ]:
# collect garbage
gc.collect()

<font size="3"> Do some pre-processing for SpaGE:

In [ ]:
# 1) Filter cells with < 10 expressed genes (counts > 0)
# Scanpy expects "min_genes" to count number of genes per cell (non-zero entries)
sc.pp.filter_cells(adata_sc, min_genes=20)   # modifies adata_sc in-place
print(f"After sc.pp.filter_cells: {adata_sc.n_obs} cells, {adata_sc.n_vars} genes")

# 2) Per-cell normalization to counts-per-million (CPM) and log1p
# normalize_total works row-wise (per observation). target_sum=1e6 replicates your earlier scale.
sc.pp.normalize_total(adata_sc, target_sum=1e6, inplace=True)  # scales rows so row-sum == 1e6
sc.pp.log1p(adata_sc)  # in-place log(1 + X). Works with sparse X.
print("Applied per-cell normalize_total(target_sum=1e6) + log1p.")


# Optional step: memory efficiency purpose only!
# Compute HVGs using Seurat v3 flavor. This writes adata_sc.var['highly_variable'].
# You can change flavor to 'cell_ranger' or 'seurat_v3' depending on preference.
sc.pp.highly_variable_genes(adata_sc, n_top_genes=10000, flavor='seurat_v3', subset=False, inplace=True)
# Subset to keep only HVGs (creates a smaller AnnData)
hv_mask = adata_sc.var['highly_variable'].values.astype(bool)
print(f"Found {hv_mask.sum()} HVGs (requested {10000}). Subsetting AnnData...")
adata_sc = adata_sc[:, hv_mask].copy()




<div style="border: 2px solid #c1121f; padding: 15px; border-radius: 8px;">
<ul style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #c1121f; list-style-type: none; font-weight: bold;">
    <li>How many genes do the spatial and single-cell data set have in common?</li>
    <li>How many genes are measured in the single-cell dataset that are not measured in the spatial dataset?
</ul>
</div>

In [ ]:
## use .intersection
gene_intersection = ....

## use a set operation
genes_not_intersection = ....

print(f"Number of genes measured both in the sc and spatial datasets: {len(gene_intersection)}")
print(f"Number of genes measured in the sc reference only (after HVG): {len(genes_not_intersection)}")

In [ ]:
sc.pp.normalize_total(adata_section, target_sum=1_000_000, inplace=True)
sc.pp.log1p(adata_section)

<font size="3"> Let's find some brain-specific genes to predict with SpaGE

In [ ]:
brain_region_specific_genes = [
    "Bcl11b", "Cux2", "Fezf2", "Tbr1", "Satb2", # isocortex
    "Foxp2",  # Striatum specific
    "Penk",   # Basal ganglia specific
    "Zic1",   # Cerebellum specific
    "Hoxb2",  # Hindbrain specific
    "Dlx6",   # Subpallium specific
]

<font size="3"> Let's split them in overlap_genes (measured in spatial dataset and sc reference) and non_overlap_genes (unmeasured in the spatial dataset)

In [ ]:
# Checking for genes from brain_region_specific_genes that are measured by spatial data
overlap_genes = list(set(list(adata_section.var.index)) & set(brain_region_specific_genes))
overlap_genes

In [ ]:
# Checking for genes from brain_region_specific_genes that are NOT measured by spatial data
non_overlap_genes = list(set(brain_region_specific_genes) - set(overlap_genes))
non_overlap_genes

### Leave-one-gene-out cross validation

<font size="3"> Since ground-truth is available, we can leave out the measured (spatial) genes, predict their expression with SpaGE and then compare it to the real expression.

In [ ]:
# sc df
RNA_data = adata_sc.to_df()
RNA_data = RNA_data.T

# spatial
spatial_data = adata_section.to_df()

<font size="3"> Define the function to predict genes measured in the spatial dataset to have a ground truth and plot the measured expression next to the predicted one.

``` python

def predicted_GE_LOOCV(Gene_set, spatial_data, RNA_data):
    
    # Initialize an empty DataFrame to store the predicted values
    predicted_df = pd.DataFrame(index=spatial_data.index)

    for i in Gene_set:
        # Predict the expression of the current gene using the SpaGE method
        Imp_Genes = SpaGE(spatial_data.drop(i, axis=1), RNA_data.T, n_pv=30, genes_to_predict=[i])
        Imp_Genes.index = spatial_data.index
        
        # Add the predicted values for the current gene as a new column in the DataFrame
        predicted_df[i] = Imp_Genes[i]

    return predicted_df
```

In [ ]:
predicted_df_loocv = pd.read_csv(f"SpaGE_data/predicted_GE_loocv_SpaGE.csv",index_col=0)
predicted_df_loocv

<font size="3"> Since we have a ground truth, create a Series to store Spearman correlation results for each gene.

<font size="3"> Our gene_set in this case is a set of genes shared between the spatial and the sc datasets.

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

plt.style.use('dark_background')

def plot_measured_vs_predicted_spatial(Gene_set, spatial_data, predicted_df, adata_section):
    """
    Plot measured and predicted spatial expression for genes, with calculated Spearman correlations.
    
    Parameters:
        Gene_set (list): List of genes to visualize.
        spatial_data (DataFrame): Measured expression data (columns are genes, rows are spots).
        predicted_df (DataFrame): Predicted expression data (columns are genes, rows are spots).
        adata_section (AnnData): Spatial data with 'x' and 'y' coordinates.
    
    Returns:
        dict: Spearman correlations for each gene.
    """
    correlations = {}  
    
    for gene in Gene_set:
        # Calculate Spearman correlation
        correlation = spearmanr(spatial_data[gene], predicted_df[gene])[0]
        correlations[gene] = correlation
        
        # Create a figure with two scatterplots
        fig = plt.figure(figsize=(15, 8))  # Adjust figure size
        gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.3)  # Create grid spec for layout
        
        # --- Measured Gene Expression ---
        ax0 = fig.add_subplot(gs[0])
        cmap_measured = spatial_data[gene]
        cmap_measured = np.clip(cmap_measured, None, np.percentile(cmap_measured, 99))  # Clip to 99th percentile
        sc1 = ax0.scatter(adata_section.obs['x'], -adata_section.obs['y'], s=1, c=cmap_measured, cmap="viridis", alpha=1)
        
        ax0.set_title(f"Measured Expression: {gene}", fontsize=12)
        ax0.set_xlabel("X Coordinate")
        ax0.set_ylabel("Y Coordinate")
        ax0.set_aspect("equal", adjustable="box")
        ax0.grid(False)
        
        # --- Predicted Gene Expression ---
        ax1 = fig.add_subplot(gs[1])
        cmap_predicted = predicted_df_loocv[gene]
        cmap_predicted = np.clip(cmap_predicted, None, np.percentile(cmap_predicted, 99))  # Clip to 99th percentile
        sc2 = ax1.scatter(adata_section.obs['x'], -adata_section.obs['y'], s=1, c=cmap_predicted, cmap="viridis", alpha=1)
        
        ax1.set_title(f"Predicted Expression: {gene}", fontsize=12)
        ax1.set_xlabel("X Coordinate")
        ax1.set_ylabel("Y Coordinate")
        ax1.set_aspect("equal", adjustable="box")
        ax1.grid(False)
        
        # --- Colorbar ---
        cbar_ax = fig.add_subplot(gs[2])  # Third subplot for color bar
        cbar = fig.colorbar(sc1, cax=cbar_ax)
        cbar.set_label("Expression")
        
        # Adjust layout and show
        plt.tight_layout()
        plt.show()
    
    return correlations


In [ ]:
correlations = plot_measured_vs_predicted_spatial(overlap_genes, spatial_data, predicted_df_loocv, adata_section)

<font size="3"> Let's take a look at the Spearman Correlations between measured and predicted gene expression:

In [ ]:
# Since we have a ground truth, create a Series to store Spearman correlation results for each gene
# Our gene_set in this case is a set of genes shared between the spatial and the sc datasets
correlations_spage = pd.DataFrame(list(correlations.items()), columns=["Gene", "correlations_spage"]).set_index("Gene")

In [ ]:
correlations_spage

### Predict unmeasured genes

<font size="3">  Define a function to predict unmeasured genes in the spatial dataset. 

<font size="3">  In this case correlations cannot be calculated as there is no ground truth available.

```python

def predicted_unmeasured_GE(Gene_set, spatial_data, RNA_data):
    
    # Initialize an empty DataFrame to store the predicted values
    predicted_df = pd.DataFrame(index=spatial_data.index)

    for i in Gene_set:
        # Predict the expression of the current gene using the SpaGE method
        Imp_Genes = SpaGE(spatial_data, RNA_data.T, n_pv=30, genes_to_predict=[i])
        Imp_Genes.index = spatial_data.index
        
        # Add the predicted values for the current gene as a new column in the DataFrame
        predicted_df[i] = Imp_Genes[i]

    return predicted_df
```

In [ ]:
predicted_df = pd.read_csv(f"SpaGE_data/predicted_unmeasured_GE_SpaGE.csv",index_col=0)

In [ ]:
def plot_predicted_spatial(Gene_set, predicted_df, adata_section):
    """
    Plot predicted spatial expression for genes.
    
    Parameters:
        Gene_set (list): List of genes to visualize.
        predicted_df (DataFrame): Predicted expression data (columns are genes, rows are spots).
        adata_section (AnnData): Spatial data with 'x' and 'y' coordinates.
    
    Returns:
        None
    """
    for gene in Gene_set:
        # Create a figure for the predicted expression
        fig, ax = plt.subplots(figsize=(8, 6))  # Adjust figure size
        
        # Get predicted values and clip to the 99th percentile
        cmap_predicted = predicted_df[gene]
        cmap_predicted = np.clip(cmap_predicted, None, np.percentile(cmap_predicted, 99))
        
        # Plot the predicted expression
        scatter = ax.scatter(
            adata_section.obs['x'], -adata_section.obs['y'], 
            s=1, c=cmap_predicted, cmap="viridis", alpha=1
        )
        
        # Add title and labels
        ax.set_title(f"Predicted Expression: {gene}", fontsize=14)
        ax.set_xlabel("X Coordinate")
        ax.set_ylabel("Y Coordinate")
        ax.set_aspect("equal", adjustable="box")
        ax.grid(False)
        
        # Add a colorbar
        cbar = plt.colorbar(scatter, ax=ax, pad=0.01)
        cbar.set_label("Expression")
        
        # Adjust layout and show
        plt.tight_layout()
        plt.show()

In [ ]:
# Define the set of genes (from the non_overlap_genes list) to be predicted by SpaGE
plot_predicted_spatial(non_overlap_genes, predicted_df, adata_section)

<div style="border: 2px solid #c1121f; padding: 15px; border-radius: 8px;">
<ul style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #c1121f; list-style-type: none; font-weight: bold;">
    <li> There is no definitive ground truth for the real gene expression profiles of these genes. However, an alternative approach could be to plot region-specific genes and verify whether their expression aligns with the expected brain regions.</li>
    <li>For example, Tbr1 is known to be Isocortex-specific; does its expression show higher levels in the Isocortex? Similarly, Foxp2 is Striatum-specific; does it exhibit a Striatum-specific expression pattern?<li>
<li> Use <a href="https://mouse.brain-map.org/search/index" target="_blank" style="color: #c1121f; text-decoration: underline;">this resource</a> to check region-specific gene expression.</li></li>
</ul>
</div>

<font size="3"> Let's delete some objects we won't be using anymore to free some RAM:

In [ ]:
del RNA_data, spatial_data

In [ ]:
# collect garbage
gc.collect()

# Unmeasured gene imputation with Tangram
[Deep learning and alignment of spatially resolved single-cell transcriptomes with Tangram (Biancalani et al. 2021)](https://www.nature.com/articles/s41592-021-01264-7)

In [ ]:
import tangram as tg

import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")

#### Common gene set between reference and spatial dataset

<font size="3"> We want to select our training genes. These genes are shared between the two datasets and should capture the biological variance between cell types. For this, we first compute marker genes on the single-cell data and then use Tangram’s preprocessing function to subset to those genes that are also present in the spatial data.

In [ ]:
markers = list(set.intersection(set(adata_sc.var_names), set(adata_section.var_names)))
len(markers)

<font size="3"> `tg.pp_adatas` computes the overlap between single-cell data and spatial data on the list of genes provided in the genes argument, it then stores the resulting gene set under 'training_genes' in both adata objects under the `.uns` key and enforces consistent ordering of the genes.

In [ ]:
tg.pp_adatas(adata_sc, adata_section, genes=markers) # using the same single-cell reference that we used to run SpaGE

<font size="3"> Check that the function performs as we expect:

In [ ]:
assert "training_genes" in adata_sc.uns
assert "training_genes" in adata_section.uns

print(f"Number of training_genes: {len(adata_sc.uns['training_genes'])}")

#### Computing the map from single-cells to spatial voxels

<font size="3"> Having specified the training genes, we can now create the map from dissociated single-cell measurements to the spatial locations. For this, we are going to use the `map_cells_to_space` function from tangram. This function has two different modes, `mode='cells'` and `mode='clusters'`. The latter only maps averaged single-cells which makes the mapping computationally faster and more robust when mapping between specimen.

```python
ad_map = tg.map_cells_to_space(
    adata_sc,
    adata_section,
    mode="cells",
    density_prior="rna_count_based",
    num_epochs=500,
    device="cpu",  # or: cpu
)


<font size="3"> We run `tg.map_cells_to_space` for you (takes a while to run).

<font size="3"> `ad_map` is Tangram’s mapping from cell i to spatial voxels j is stored in the .X property of `ad_map`. Size = number of spots in the spatial data object x number of cells in the sc reference dataset.
<font size="3">  The meaning of the .var and .obs also changes:

- <font size="3"> in `.var` we have the available metadata of the spatial data, adata_section
- <font size="3"> in `.obs` we have the available metadata of the single-cell data, adata_sc
- <font size="3"> In addition, the information about the training run is stored in the .uns key, see `.uns['training_genes_df']` and `.uns[training_history]`.

<font size="3"> Now we can **project the genes present in the single-cell data to the spatial locations**. This is easily achieved by multiplying the mapping matrix stored in ad_map with the original single-cell data stored in adata_sc. Tangram already provides a convenience function which takes in a mapping and the corresponding single-cell data. The result is a **spatial voxel by genes matrix** which technically is identical to the original spatial data adata_section but **contains expression values for all genes**.

```python
ad_ge = tg.project_genes(adata_map=ad_map, adata_sc=adata_sc)
ad_ge
```

<font size="3"> We run `tg.project_genes` for you, let's just read the resulting AnnData object

In [ ]:
ad_ge = ad.read_h5ad(f"Tangram_data/ad_ge.h5ad")

## add .obsm['X_spatial'] so we can use sc.pl.spatial to plot gene expression
ad_ge.obsm['X_spatial'] = np.array(ad_ge.obs[['x','y']])

# fix index names
ad_ge.var.reset_index(inplace=True)
ad_ge.var.set_index('index_renamed', inplace=True)
ad_ge.var.index = [str(gene).capitalize() for gene in ad_ge.var.index]
ad_ge.var

<font size="3"> Next, we will compare the new spatial data with the original measurements.

<font size="3"> To do this we will plot the true expression and the predicted expression of the shared genes, as well as the Spearman correlation between the true and predicted expression in all the spots.

<font size="3"> Plot the true expression from `adata_section`

In [ ]:
adata_section.var.index = [str(gene).capitalize() for gene in adata_section.var.index]

In [ ]:
from scipy.sparse import csr_array, csr_matrix, issparse

gt_expr = adata_section[:, adata_section.var.index.isin([overlap_genes])].X

In [ ]:
from scipy.sparse import issparse
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')

for gene in overlap_genes:
    fig, axs = plt.subplots(1, 3, figsize=(20, 8))  # 3 columns

    # --- Extract measured (ground truth) expression ---
    gt_expr = adata_section[:, adata_section.var.index.isin([gene])].X
    gt_expr = np.clip(gt_expr, None, np.percentile(gt_expr, 99))
    gt_vmin, gt_vmax = gt_expr.min(), gt_expr.max()

    # --- Extract Tangram expression ---
    ad_ge_sub = ad_ge[:, ad_ge.var.index.isin([gene])]
    ad_ge_expr = ad_ge_sub.X
    ad_ge_expr = np.clip(ad_ge_expr, None, np.percentile(ad_ge_expr, 99))

    # --- Extract SpaGE expression ---
    spage_expr = predicted_df_loocv[gene].values
    spage_expr = np.clip(spage_expr, None, np.percentile(spage_expr, 99))

    # --- Shared color scale for Tangram & SpaGE ---
    pred_vmin = min(ad_ge_expr.min(), spage_expr.min())
    pred_vmax = max(ad_ge_expr.max(), spage_expr.max())

    # --- Ground truth ---
    sc0 = axs[0].scatter(
        adata_section.obs['x'], adata_section.obs['y'],
        c=gt_expr, cmap='viridis', s=1, vmin=gt_vmin, vmax=gt_vmax
    )
    axs[0].invert_yaxis()
    axs[0].set_title(f'Measured: {gene}', fontsize=14, pad=15)
    axs[0].grid(False)
    fig.colorbar(sc0, ax=axs[0], pad=0.01)

    # --- Tangram ---
    sc1 = axs[1].scatter(
        adata_section.obs['x'], adata_section.obs['y'],
        c=ad_ge_expr, cmap='viridis', s=1, vmin=pred_vmin, vmax=pred_vmax
    )
    axs[1].invert_yaxis()
    axs[1].set_title(f'Tangram: {gene}', fontsize=14, pad=15)
    axs[1].grid(False)
    fig.colorbar(sc1, ax=axs[1], pad=0.01)

    # --- SpaGE ---
    sc2 = axs[2].scatter(
        adata_section.obs['x'], adata_section.obs['y'],
        c=spage_expr, cmap='viridis', s=1, vmin=pred_vmin, vmax=pred_vmax
    )
    axs[2].invert_yaxis()
    axs[2].set_title(f'SpaGE: {gene}', fontsize=14, pad=15)
    axs[2].grid(False)
    fig.colorbar(sc2, ax=axs[2], pad=0.01)

    # Adjust layout
    plt.subplots_adjust(top=0.85, wspace=0.3)
    plt.show()


<font size="3"> Plot the predicted expression from `ad_ge`

In [ ]:
plt.style.use('dark_background')

for gene in non_overlap_genes:
    fig, axs = plt.subplots(1, 2, figsize=(15, 8))

    # --- Extract Tangram expression ---
    ad_ge_sub = ad_ge[:, ad_ge.var.index.isin([gene])]
    ad_ge_expr = ad_ge_sub.X
    ad_ge_expr = np.clip(ad_ge_expr, None, np.percentile(ad_ge_expr, 99))

    # --- Extract SpaGE expression ---
    spage_expr = predicted_df[gene].values
    spage_expr = np.clip(spage_expr, None, np.percentile(spage_expr, 99))

    # --- Determine shared color scale ---
    vmin = min(ad_ge_expr.min(), spage_expr.min())
    vmax = max(ad_ge_expr.max(), spage_expr.max())

    # --- Tangram ---
    sc0 = axs[0].scatter(
        adata_section.obs['x'], adata_section.obs['y'],
        c=ad_ge_expr, cmap='viridis', s=1, vmin=vmin, vmax=vmax
    )
    axs[0].invert_yaxis()
    axs[0].set_title(f'Tangram: {gene}', fontsize=14, pad=15)
    axs[0].grid(False)
    fig.colorbar(sc0, ax=axs[0], pad=0.01)

    # --- SpaGE ---
    sc1 = axs[1].scatter(
        adata_section.obs['x'], adata_section.obs['y'],
        c=spage_expr, cmap='viridis', s=1, vmin=vmin, vmax=vmax
    )
    axs[1].invert_yaxis()
    axs[1].set_title(f'SpaGE: {gene}', fontsize=14, pad=15)
    axs[1].grid(False)
    fig.colorbar(sc1, ax=axs[1], pad=0.01)

    # Ensure titles and layout are visible
    plt.subplots_adjust(top=0.85, wspace=0.3)
    plt.show()


<font size="3"> Calculate Spearman correlations

In [ ]:
# Subset the AnnData objects to include only the genes in overlap_genes
# Assuming both AnnData objects have the same gene ordering
ad_true  = adata_section[:, adata_section.var_names.isin(overlap_genes)]
ad_ge_predicted = ad_ge[:, ad_ge.var_names.isin(overlap_genes)]

# Initialize an empty dictionary to store correlations
correlation_dict = {}

# Loop over each gene to calculate Spearman correlation
for gene in overlap_genes:
    # Extract gene expression vectors for the current gene
    expr1 = ad_true[:, gene].X.flatten()
    expr2 = ad_ge_predicted[:, gene].X.flatten()
    
    # Calculate Spearman correlation for the current gene
    corr, _ = st.spearmanr(expr1, expr2)
    
    # Store the result in the dictionary
    correlation_dict[gene] = corr

# Convert the dictionary to a DataFrame
tangram_correlations = pd.DataFrame.from_dict(correlation_dict, orient='index', columns=['correlations_tangram'])

tangram_correlations

<font size="3"> Since we predicted the same genes as SpaGE, let's compare the Spearman correlations:

<div style="border: 2px solid #c1121f; padding: 15px; border-radius: 8px;">
<ul style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #c1121f; list-style-type: none; font-weight: bold;">
    <li> Which tool results in higher correlation between the true-measured gene expression and the predicted expression of the genes of interest?</li>
</ul>
</div>

In [ ]:
all_correlations = # ..... 
all_correlations[['correlations_spage','correlations_tangram']]

<font size="3"> Let's delete some objects we won't be using anymore to free some RAM:

In [ ]:
del ad_ge

In [ ]:
del all_correlations, correlations_spage, tangram_correlations, predicted_df, ad_true, ad_ge_predicted

In [ ]:
gc.collect()

# Cell type deconvolution with cell2location
[Cell2location maps fine-grained cell types in spatial transcriptomics (Kleshchevnikov et al. 2022)](https://www.nature.com/articles/s41587-021-01139-4)

In [ ]:
#import squidpy as sq
import cell2location as c2l

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")

In [ ]:
# set raw layer as X
adata_sc.layers['raw'] = adata_sc.X

<font size="3"> Add metadata with cell type annotation to the single-cell AnnData object

In [ ]:
# read csv
metadata = pd.read_csv(f"data/cell_metadata_with_cluster_annotation.csv",index_col=0)
# filter metadata to keep only obervations in the subsetted SC AnnData object
metadata = metadata.loc[metadata.index.intersection(adata_sc.obs.index)]
# reindex rows
metadata = metadata.reindex(adata_sc.obs.index)
# assign metadata to adata_sc.obs (in this way metadata replaces all fields previously in obs)
adata_sc.obs = metadata

adata_sc.obs.head()

In [ ]:
del metadata
gc.collect()

<font size="3"> Identify the shared set of genes between spatial and single-cell 

In [ ]:
shared_features = [feature for feature in adata_section.var_names if feature in adata_sc.var_names]

# filter 
adata_sc = adata_sc[:, adata_sc.var.index.isin(shared_features)].copy()
adata_section = adata_section[:, shared_features].copy() # probably no need to filter the spatial gene set
# reorder genes to have them match between spatial data and sc
adata_sc = adata_sc[:, shared_features].copy()

adata_section.shape[1] == adata_sc.shape[1]
print(f'Number of shared genes between spatial data and reference sc: {adata_section.shape[1]}')

<font size="3"> **Fit the reference model**

<font size="3"> When your spatial transcriptomics dataset is whole-transcriptome: The default parameters cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12 are a good starting point, however, you can increase the cut-off to exclude more genes. To preserve marker genes of rare cell types we recommend low cell_count_cutoff=5, however, cell_percentage_cutoff2 and nonz_mean_cutoff can be increased to select between 8k-16k genes.

<font size="3"> In this 2D histogram, orange rectangle highlights genes excluded based on the combination of number of cells expressing that gene (Y-axis) and average RNA count for cells where the gene was detected (X-axis).

<font size="3"> This step can be skipped when a limited gene panel is available.

In [ ]:
# fit the reference model
selected = c2l.utils.filtering.filter_genes(
    adata_sc, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12
)

adata_sc = adata_sc[:, selected].copy()
adata_section = adata_section[:, selected].copy()

In [ ]:
print(f"The spatial AnnData object has {adata_section.shape[0]} obs and {adata_section.shape[1]} features.")

In [ ]:
print(f"The single-cell AnnData object has {adata_sc.shape[0]} obs and {adata_sc.shape[1]} features.")

<font size="3"> **Estimation of reference cell type signatures (NB regression)**

<font size="3"> The following two command are computationally intensive and require GPU. We run them for you. You can read the output from `stdata_deconv.h5ad`

<font size="3"> The signatures are estimated from scRNA-seq data, accounting for batch effect, using a Negative binomial regression model.

<font size="3"> First, prepare anndata object for the regression model:

``` python
c2l.models.RegressionModel.setup_anndata(
    adata=adata_sc,
    #batch_key="Replicates", # there's no batch 
    labels_key="class", # cell type = class
    #categorical_covariate_keys=["assay"],
    layer="raw",
)
```

```python
model = c2l.models.RegressionModel(adata_sc)
# default, try on GPU:
model.train(max_epochs=250, batch_size=2500, train_size=1, lr=0.002)

model.export_posterior(
    adata_sc,
    sample_kwargs={"num_samples": 1000, "batch_size": 2500},
)
```

<font size="3"> **Train the model** to estimate the reference cell type signatures:

```python
# export estimated expression in each cluster
if "means_per_cluster_mu_fg" in adata_sc.varm.keys():
    inf_aver = adata_sc.varm["means_per_cluster_mu_fg"][
        [f"means_per_cluster_mu_fg_{i}" for i in adata_sc.uns["mod"]["factor_names"]]
    ].copy()
else:
    inf_aver = adata_sc.var[
        [f"means_per_cluster_mu_fg_{i}" for i in adata_sc.uns["mod"]["factor_names"]]
    ].copy()

inf_aver.columns = adata_sc.uns["mod"]["factor_names"]
inf_aver.head()

# cell 2 location cell type mapping
print('start with Cell2location')
c2l.models.Cell2location.setup_anndata(
    adata=adata_section,
)

model = c2l.models.Cell2location(
    adata_section,
    cell_state_df=inf_aver,
    N_cells_per_location=8,
)
model.view_anndata_setup()

print('start with training')
model.train(max_epochs=30000, batch_size=None, train_size=1)#, use_gpu=use_gpu)

adata_section = model.export_posterior(
    adata_section,
    sample_kwargs={
        "num_samples": 1000,
        "batch_size": model.adata.n_obs}#,
        #"use_gpu": use_gpu,
)


adata_section.obs[adata_section.uns["mod"]["factor_names"]] = adata_section.obsm["q05_cell_abundance_w_sf"]
```

<font size="3"> Read the pre-computed AnnData object with cell2location results

In [ ]:
adata_deconv = ad.read_h5ad(f"cell2location_data/stdata_deconv.h5ad")

In [ ]:
### cell2location result
adata_deconv.obs[adata_section.obs['class'].unique()]

<font size="3"> Visualize cell abundance in spatial coordinates:

In [ ]:
sc.pl.spatial(adata_deconv, color=['01 IT-ET Glut','04 DG-IMN Glut','30 Astro-Epen'],spot_size=0.04)

In [ ]:
# collect garbage
gc.collect()